# Quandela Cloud First-Run Demo

This notebook runs a simple photonic experiment once on Quandela Cloud.

1. Enter your token.
2. Build an experiment with a single beam splitter.
3. Run it on the free simulator `sim:belenos`.
4. (Optional) Run it on the real QPU `qpu:belenos`.

**Before you start**

- A token issued from **Tokens** on the Cloud site (see the user manual)
- To run on the QPU: QPU time and access to the `qpu:belenos` platform, both assigned by your administrator

In [ ]:
import perceval as pcvl
print("Perceval", pcvl.__version__)

## 1. Enter your token

Running the cell below shows an input box. Paste the token you copied with **Copy** on the Cloud **Tokens** page and press Enter.
A token works like a password, so it is entered here instead of being written into the notebook.

In [ ]:
from getpass import getpass

TOKEN = getpass("Paste your token: ").strip()
print("Token length:", len(TOKEN))

## 2. Build the experiment

Send one photon into each input of a single beam splitter (BS), i.e. the input state |1,1⟩.
When the two photons are indistinguishable they always leave through the same output (the Hong-Ou-Mandel effect), so the results should be mostly |2,0⟩ and |0,2⟩, with almost no |1,1⟩.

In [ ]:
from perceval import BS, BasicState

circuit = BS()
input_state = BasicState([1, 1])

pcvl.pdisplay(circuit)

## 3. Helper functions

`run_on_cloud` takes a platform name and a number of shots, sends the job to the Cloud and returns the results.
`job_name` is the name shown on the Cloud **Jobs** page.

In [ ]:
from perceval import RemoteProcessor
from perceval.algorithm import Sampler


def run_on_cloud(platform, shots, job_name):
    processor = RemoteProcessor(platform, token=TOKEN)
    processor.set_circuit(circuit)
    processor.with_input(input_state)
    processor.min_detected_photons_filter(1)
    # max_shots_per_call is required for remote platforms
    sampler = Sampler(processor, max_shots_per_call=shots)
    sampler.default_job_name = job_name
    return sampler.sample_count(shots)["results"]


def show(results):
    total = sum(results.values())
    for state, count in sorted(results.items(), key=lambda x: -x[1]):
        print(f"{str(state):>8}  {count:>6}  ({count / total:.1%})")

## 4. Run on the simulator (free)

`sim:belenos` is a simulator that mimics the `qpu:belenos` QPU, and it does not consume time.
The job can take from a few tens of seconds to a few minutes.

In [ ]:
sim_results = run_on_cloud("sim:belenos", shots=1000, job_name="demo_sim")
show(sim_results)

## 5. Run on the real QPU (optional)

**Caution:** Running on the QPU consumes your assigned QPU time. Depending on the queue, results can take a while.

To run it, change `RUN_QPU` to `True` in the cell below and run the cell.

In [ ]:
RUN_QPU = False

if RUN_QPU:
    qpu_results = run_on_cloud("qpu:belenos", shots=1000, job_name="demo_qpu")
    show(qpu_results)
else:
    print("QPU run skipped. Set RUN_QPU = True to run it.")

## 6. Check the results

You can check the status and time used by the `demo_sim` and `demo_qpu` jobs under **Jobs** in the left menu of the Cloud site.

For more, see the Remote Computing tutorial in the Perceval documentation (https://perceval.quandela.net/docs).